# Split Point-Cloud Viewer (diagnostic tool)

Compare **two point clouds side by side** in a desktop (tkinter) window: five spatial
views each, per-field statistics, colouring by any data field, and value/range
filtering. Built to answer questions like *"which fields should be merged?"* and
*"how do the class labels of dataset A map onto dataset B?"* — the questions the
`class_unifier` stage exists to settle.

**Per-panel interactions**

- **Fields shown…** — pick which fields the statistics box lists (all shown by
  default; untick the constant/uninformative ones to cut the clutter).
- **3D view** — open the **currently filtered selection** in a separate orbitable
  window (see below). 2-D stays the default view.
- **Histogram** / **Unique…** — pop up a histogram, or a sortable *value / count /
  %* table, of the current **Colour by** field. Each pop-up has a *respect active
  filter* toggle (off = whole cloud). Use **Unique…** to discover which of a wide
  id field (e.g. `tree_ID`, 0…~60 000) actually exist before filtering to one.
- **Mouse wheel** — zoom the spatial view under the cursor; the matplotlib toolbar
  Home button resets it.
- For `.las`/`.laz`, the info box header line also shows the **LAS version** and
  point-format id, e.g. `LAS/LAZ  1.4 (PF 6)`.

**The 3-D view**

`misc/view_cloud_3d.py` renders the selection with **PyVista + VTK** in its own
window: left-drag rotates, scroll zooms, middle-drag pans, `r` resets the camera,
`d` toggles depth shading, `q` closes.

- It runs as a **separate process**, so the 2-D window stays responsive and Cloud A
  and Cloud B can each have a 3-D window open at the same time.
- It shows the points the filter selected, capped by its own **3D max pts** budget
  (default 500 000 — the GPU takes far more than the 2-D scatter).
- Colours match the 2-D views exactly (same `color_spec` rule): a discrete class
  legend for ≤ 20 integer labels, otherwise a viridis scalar bar.
- Requires `pyvista` + `vtk`. If they are missing the button explains how to install
  them. *(Open3D is not used: it publishes no wheels for this env's Python 3.13.)*

**⚠️ Integer columns colour as classes, float columns do not**

The rule behind every colour decision here is
`discrete = np.issubdtype(dtype, np.integer) and n_unique <= 20`. A label column that
arrives as **float** can never satisfy it, so it is drawn as a continuous viridis ramp
with **no class legend**, its *Unique* cell shows `-`, and its histogram falls back to
generic bins.

That is why `class_unifier` defaults to **`output_format: ply`**: a PLY stores a dtype
per property, so `semantic_seg` stays `uint8` and `tree_ID` stays an int. Its `.npy`
option is a single float64 matrix and views poorly for exactly this reason — load the
`.ply` when you want to *see* classes.

**Supported formats**

| Format | How fields are found |
|---|---|
| `.las` / `.laz` | all point dimensions (incl. extra dims like `tree_ID`, `dist_axes`, `Class`); `x/y/z` are the scaled coordinates; header line shows version + point format |
| `.ply` | all vertex properties, each with its own dtype — **the best format to inspect** |
| `.npy` (structured array) | names taken from the array's dtype |
| `.npy` (plain N×K matrix) | names taken from a **JSON sidecar** (see below); every column is float64 |

**The `.npy` JSON sidecar convention**

A bare `.npy` matrix stores no column names, so the viewer looks for a JSON file with
the **same stem** next to it — `plot_01_007.npy` → `plot_01_007.json`:

```json
{"columns": ["x", "y", "z", "2tree_ID"]}
```

Any *additional* keys in that JSON are treated as metadata about the cloud and shown
in the info box. If the sidecar is missing, unreadable, or its `columns` count does
not match the matrix, the viewer falls back to `x, y, z, col_3, …` and shows a
warning. `class_unifier` writes these sidecars (alongside either output format), so
its exports carry their column names and full provenance.

**How to run**

1. Use the **aifor** kernel (`mamba env` with `laspy`, `plyfile`, `matplotlib`, and
   `pyvista`+`vtk` for the 3-D view).
2. Run the cells top to bottom; the last cell opens the viewer window.
3. ⚠️ The notebook kernel is **busy while the window is open** (tkinter's event loop
   runs in the kernel). Close the window to get the kernel back. The 3-D windows are
   separate processes and are *not* affected by this.

In [31]:
"""Imports.

matplotlib is embedded directly into the tkinter window via ``FigureCanvasTkAgg``:
we build ``Figure`` objects ourselves and never touch ``pyplot`` / ``plt.show()``.
Keeping pyplot (and its hidden global state) out of the picture is the recommended
way to host matplotlib inside a GUI toolkit.

PyVista/VTK (the 3-D view) is deliberately NOT imported here: it runs in a separate
process (``misc/view_cloud_3d.py``) because VTK and tkinter each want to own the
event loop. See ``CloudPanel.open_3d_view``.
"""
import importlib.util
import json
import logging
import subprocess
import sys
import tempfile
import tkinter as tk
from pathlib import Path
from tkinter import filedialog, messagebox, ttk
from tkinter.scrolledtext import ScrolledText

import laspy
import numpy as np
import pandas as pd
from matplotlib import colormaps
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
from matplotlib.figure import Figure
from matplotlib.lines import Line2D
from plyfile import PlyData

print("imports OK")

imports OK


In [32]:
# ---------------------------------------------------------------------------
# Loaders: every supported format is normalised into ONE structure so the GUI
# (and any code you write in later cells) never has to care where a cloud
# came from:
#
#   {
#     "name":       "plot_01_tree_ID_dist_axes.las",   # file name (display)
#     "path":       Path(...),                          # full path
#     "file_type":  "LAS/LAZ" | "PLY" | "NPY",
#     "num_points": 18_123_456,
#     "fields":     {"x": ndarray, "y": ndarray, ...},  # plain 1-D numpy arrays
#     "meta":       {...},   # extra sidecar keys (NPY only), shown in info box
#     "header":     {...},   # format-specific header facts (LAS version, ...)
#     "warnings":   [...],   # human-readable problems, shown in info box
#   }
#
# laspy gotcha (repo convention): laspy exposes x/y/z as ScaledArrayView and
# bit fields (return_number, ...) as SubFieldView. Those are NOT plain arrays
# and confuse numpy/pandas, so every accessor is materialised with np.asarray.
# ---------------------------------------------------------------------------

# Raw integer coordinate dimensions of the LAS spec. We drop them and expose the
# scaled x/y/z instead (raw ints are scale*X+offset away from real coordinates).
_LAS_RAW_COORDS = ("X", "Y", "Z")


def _load_las(path: Path) -> dict:
    """Read a .las/.laz file into the common cloud structure."""
    las = laspy.read(str(path))
    fields = {
        "x": np.asarray(las.x, dtype=np.float64),
        "y": np.asarray(las.y, dtype=np.float64),
        "z": np.asarray(las.z, dtype=np.float64),
    }
    # Every remaining dimension, including extra-bytes dims (tree_ID, dist_axes,
    # Class, ...) that 3DFin / forainet_prep attach.
    for dim in las.point_format.dimension_names:
        if dim in _LAS_RAW_COORDS:
            continue
        fields[dim] = np.asarray(getattr(las, dim))
    return {
        "name": path.name,
        "path": path,
        "file_type": "LAS/LAZ",
        "num_points": len(fields["x"]),
        "fields": fields,
        "meta": {},
        # Header facts worth surfacing in the info box. Same idiom as
        # pipeline/forainet_prep.py: version as a string ("1.4"), point format id.
        "header": {
            "las_version": str(las.header.version),
            "point_format": int(las.point_format.id),
        },
        "warnings": [],
    }


def _load_ply(path: Path) -> dict:
    """Read a .ply file (all vertex properties) into the common cloud structure."""
    ply = PlyData.read(str(path))
    vertex = ply["vertex"].data
    fields = {name: np.asarray(vertex[name]) for name in vertex.dtype.names}
    return {
        "name": path.name,
        "path": path,
        "file_type": "PLY",
        "num_points": len(vertex),
        "fields": fields,
        "meta": {},
        "header": {},
        "warnings": [],
    }


def _npy_column_names(path: Path, n_cols: int) -> tuple:
    """Resolve column names for a bare N x K .npy matrix.

    Looks for a JSON sidecar with the same stem (plot_01_007.npy ->
    plot_01_007.json) holding at least {"columns": [...]}. Every OTHER key of
    that JSON is returned as metadata so the GUI can display it. Falls back to
    generic names (x, y, z, col_3, ...) whenever the sidecar is unusable, and
    reports what happened through the warnings list.

    Returns (names, meta, warnings).
    """
    warnings, meta = [], {}
    names = None

    sidecar = path.with_suffix(".json")
    if sidecar.exists():
        try:
            info = json.loads(sidecar.read_text(encoding="utf-8"))
            cols = info.get("columns")
            if isinstance(cols, list) and len(cols) == n_cols:
                names = [str(c) for c in cols]
                meta = {k: v for k, v in info.items() if k != "columns"}
            else:
                got = len(cols) if isinstance(cols, list) else repr(cols)
                warnings.append(
                    f"sidecar {sidecar.name}: 'columns' does not match the array "
                    f"({got} names vs {n_cols} columns)"
                )
        except (OSError, json.JSONDecodeError) as exc:
            warnings.append(f"sidecar {sidecar.name} could not be read: {exc}")
    else:
        warnings.append(f"no sidecar {sidecar.name} next to the array")

    if names is None:
        names = ["x", "y", "z"][:n_cols] + [f"col_{i}" for i in range(3, n_cols)]
        warnings.append("using fallback column names: " + ", ".join(names))
    return names, meta, warnings


def _load_npy(path: Path) -> dict:
    """Read a .npy file (structured array or bare N x K matrix)."""
    arr = np.load(str(path), allow_pickle=False)
    meta, warnings = {}, []

    if arr.dtype.names:
        # Structured array: numpy already stored the column names for us.
        fields = {name: np.asarray(arr[name]) for name in arr.dtype.names}
    else:
        if arr.ndim == 1:  # a single column still gets the full treatment
            arr = arr.reshape(-1, 1)
        if arr.ndim != 2:
            raise ValueError(f"{path.name}: expected an N x K matrix, got shape {arr.shape}")
        names, meta, warnings = _npy_column_names(path, arr.shape[1])
        # ascontiguousarray: column slices of a 2-D array are strided views;
        # copying them keeps all downstream numpy ops fast and predictable.
        fields = {name: np.ascontiguousarray(arr[:, i]) for i, name in enumerate(names)}

    n = len(next(iter(fields.values())))
    return {
        "name": path.name,
        "path": path,
        "file_type": "NPY",
        "num_points": n,
        "fields": fields,
        "meta": meta,
        "header": {},
        "warnings": warnings,
    }


_LOADERS = {".las": _load_las, ".laz": _load_las, ".ply": _load_ply, ".npy": _load_npy}


def load_point_cloud(path) -> dict:
    """Load any supported point-cloud file into the common structure."""
    path = Path(path)
    loader = _LOADERS.get(path.suffix.lower())
    if loader is None:
        raise ValueError(
            f"unsupported file type '{path.suffix}' "
            f"(supported: {', '.join(sorted(_LOADERS))})"
        )
    return loader(path)


def summarize_fields(cloud: dict, fields=None) -> pd.DataFrame:
    """Per-field statistics table: dtype, Min, Max, Unique.

    ``fields`` optionally restricts (and orders) the rows to a list of field
    names; ``None`` (default) reports every field in insertion order. Unknown
    names are skipped so a stale selection never raises.

    Unique counts are only computed for integer fields (labels, ids): they are
    the interesting ones for label diagnosis, and np.unique on huge float
    columns would be slow for no benefit.
    """
    names = list(cloud["fields"]) if fields is None else fields
    rows = []
    for name in names:
        col = cloud["fields"].get(name)
        if col is None:
            continue
        row = {"Field": name, "dtype": str(col.dtype), "Min": "-", "Max": "-", "Unique": "-"}
        if np.issubdtype(col.dtype, np.number):
            row["Min"] = f"{col.min():g}"
            row["Max"] = f"{col.max():g}"
            if np.issubdtype(col.dtype, np.integer):
                row["Unique"] = f"{np.unique(col).size:,}"
        rows.append(row)
    return pd.DataFrame(rows)


def cloud_info_text(cloud: dict, fields=None) -> str:
    """The full plain-text report shown in a panel's info box.

    ``fields`` restricts which per-field stat rows are shown (see
    ``summarize_fields``); the header line, sidecar metadata and warnings are
    always shown. For LAS/LAZ the header line also carries the file version and
    point-format id, e.g. ``LAS/LAZ  1.4 (PF 6)``.
    """
    header = cloud["file_type"]
    hdr = cloud.get("header") or {}
    if hdr.get("las_version"):
        header += f"  {hdr['las_version']}"
        if hdr.get("point_format") is not None:
            header += f" (PF {hdr['point_format']})"
    lines = [
        f"{header}  |  {cloud['num_points']:,} points",
        str(cloud["path"]),
        "",
        summarize_fields(cloud, fields).to_string(index=False),
    ]
    if cloud["meta"]:
        lines += ["", "Sidecar metadata:"]
        lines += [f"  {k}: {v}" for k, v in cloud["meta"].items()]
    if cloud["warnings"]:
        lines += ["", "WARNINGS:"]
        lines += [f"  ! {w}" for w in cloud["warnings"]]
    return "\n".join(lines)


print("loaders defined")

loaders defined


In [33]:
# ---------------------------------------------------------------------------
# Filtering + projection + colour helpers.
#
# These are pure functions (no GUI, no globals) on purpose: the tkinter class
# below only wires widgets to them, and they can be reused / unit-tested from
# plain notebook cells.
# ---------------------------------------------------------------------------


def parse_values(text: str) -> list:
    """Parse a comma-separated values string ("2, 3, 4.5") into numbers.

    Each item is tried as int first, then float, else kept as a string (useful
    should a field ever hold non-numeric data). Empty items are skipped.
    """
    values = []
    for item in text.split(","):
        item = item.strip()
        if not item:
            continue
        try:
            values.append(int(item))
        except ValueError:
            try:
                values.append(float(item))
            except ValueError:
                values.append(item)
    return values


def make_mask(fields: dict, field: str, mode: str,
              vmin=None, vmax=None, values=None) -> np.ndarray:
    """Boolean keep-mask over all points of a cloud.

    mode="range":  keep points with vmin <= field <= vmax
                   (either bound may be None = unbounded on that side).
    mode="values": keep points whose field is in `values` (exact match),
                   e.g. Class in {2, 3}.
    """
    col = fields[field]
    if mode == "range":
        mask = np.ones(len(col), dtype=bool)
        if vmin is not None:
            mask &= col >= vmin
        if vmax is not None:
            mask &= col <= vmax
        return mask
    if mode == "values":
        if not values:
            return np.ones(len(col), dtype=bool)
        return np.isin(col, values)
    raise ValueError(f"unknown filter mode: {mode!r}")


def color_spec(cvals: np.ndarray, max_classes: int = 20) -> dict:
    """Decide how a value column should be coloured.

    ONE rule shared by the 2-D scatter and the 3-D view, so the two can never
    disagree about what a colour means:

    * few distinct integers (<= max_classes) -> **discrete** tab20 colours plus a
      legend, the right thing for class labels and small id sets;
    * anything else -> a **continuous** viridis ramp with a colourbar.

    Returns a dict with:
        discrete : bool
        uniq     : the distinct values (discrete only, else None)
        palette  : (K, 4) RGBA floats, one per uniq value (discrete only)
        rgba     : (N, 4) RGBA floats, one per point (discrete only)
        rgb      : (N, 3) uint8, the same colours 0-255 (discrete only) --
                   what VTK/PyVista wants
        scalars  : (N,) float64 raw values (continuous only, else None)
    """
    cvals = np.asarray(cvals)
    discrete = (np.issubdtype(cvals.dtype, np.integer)
                and np.unique(cvals).size <= max_classes)
    if not discrete:
        return {"discrete": False, "uniq": None, "palette": None,
                "rgba": None, "rgb": None, "scalars": cvals.astype(np.float64)}

    uniq = np.unique(cvals)
    tab20 = colormaps["tab20"]
    palette = np.array([tab20(i % 20) for i in range(uniq.size)])
    rgba = palette[np.searchsorted(uniq, cvals)]
    return {
        "discrete": True,
        "uniq": uniq,
        "palette": palette,
        "rgba": rgba,
        "rgb": (rgba[:, :3] * 255).round().astype(np.uint8),
        "scalars": None,
    }


# The five spatial views. The isometric ones rotate the cloud about the z axis
# by the given angle and then look at it from the front (rotated-x vs z), i.e.
# a "camera walking around the plot" at 45 deg steps between the axis-aligned
# front/side views.
VIEWS = (
    ("XY (top)", "xy"),
    ("XZ (front)", "xz"),
    ("YZ (side)", "yz"),
    ("ISO 45\N{DEGREE SIGN}", "iso45"),
    ("ISO 135\N{DEGREE SIGN}", "iso135"),
)


def project(x: np.ndarray, y: np.ndarray, z: np.ndarray, view: str):
    """Project 3-D points onto the 2-D plane of the requested view.

    Returns (u, v): the horizontal and vertical plot coordinates.
    """
    if view == "xy":
        return x, y
    if view == "xz":
        return x, z
    if view == "yz":
        return y, z
    if view in ("iso45", "iso135"):
        theta = np.deg2rad(45.0 if view == "iso45" else 135.0)
        return x * np.cos(theta) + y * np.sin(theta), z
    raise ValueError(f"unknown view: {view!r}")


print("helpers defined")

helpers defined


---
## Launch the viewer

Running the next cell opens the desktop window. **The kernel stays busy until you
close the window** — that is normal for a tkinter GUI inside a notebook.

Tips:
- Load a different file at any time with **Load…** (each panel is independent).
- **Mouse-wheel** over any spatial view zooms it at the cursor; the **matplotlib
  toolbar** under each figure gives box-zoom / pan / Home / save-as-PNG.
- **3D view** opens the current selection in its own window: drag = rotate,
  scroll = zoom, middle-drag = pan, `r` = reset, `d` = depth shading, `q` = close.
  It is a separate process, so this window keeps working and both panels can have a
  3-D window open at once. Filter first, then open it — it shows what the filter
  selected, up to **3D max pts**.
- **Histogram** and **Unique…** open a pop-up for the current **Colour by** field.
  Tick *respect active filter* in the pop-up to restrict it to the filtered points.
- **Unique…** is the quick way to find real ids in a wide field: colour by `tree_ID`,
  open it, read off an existing id, then **Filter → Values** that id — and then
  **3D view** to see that single tree from any angle.
- **Fields shown…** trims the statistics box — untick the constant/noise fields.
- **Filter → Values** with e.g. `2, 3` on a `Class` field shows only those labels;
  **Range** with min/max works better for continuous fields (`z`, `dist_axes`).
- Raise **Max points** for more detail (slower), lower it for speed.
- The window can be resized freely, including **after** a cloud is loaded.

In [35]:
# Opens the viewer window; the kernel is busy until the window is closed.
app = launch_viewer()